# SQL Interview — 80 Solution Validation (PostgreSQL 18)

Run **after** the new reseed notebook. This notebook executes all 80 reference solutions against your actual PostgreSQL database and produces a pass/fail report. A result count of zero is treated as a failed scenario unless explicitly allowed. It also checks targeted edge cases and writes `sql_interview_validation_report.csv`.

**Important:** Passing a query does not mathematically prove the query answers the question correctly; it checks execution, data availability and scenario invariants.


In [ ]:
import psycopg2, pandas as pd, traceback
from IPython.display import display
# Import the shared database configuration regardless of whether VS Code
# starts the kernel in coding-exercises/ or coding-exercises/sql/.
from pathlib import Path
import sys
_SQL_DIR = next((p for p in (Path.cwd(), Path.cwd() / "sql")
                 if (p / "db_connection.py").is_file()), None)
if _SQL_DIR is None:
    raise FileNotFoundError(
        "Cannot locate sql/db_connection.py. Open coding-exercises/ or its sql/ folder in VS Code."
    )
if str(_SQL_DIR) not in sys.path:
    sys.path.insert(0, str(_SQL_DIR))
from db_connection import connect, connection_settings

conn = connect(autocommit=True)
print('Connected to PostgreSQL:', conn.server_version)


In [ ]:
# Validate known edge-case invariants, independently of reference solutions.
checks = {
 'Q02 no-order customers': "SELECT COUNT(*) FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id=c.customer_id)",
 'Q05 never-ordered products': "SELECT COUNT(*) FROM products p WHERE NOT EXISTS (SELECT 1 FROM order_items oi WHERE oi.product_id=p.product_id)",
 'Q46 consecutive login dates': "SELECT COUNT(*) FROM (SELECT DISTINCT user_id, event_ts::date d FROM events WHERE event_type='login') a JOIN (SELECT DISTINCT user_id, event_ts::date d FROM events WHERE event_type='login') b ON a.user_id=b.user_id AND a.d=b.d+1",
 'Q51 duplicate normalized emails': "SELECT COUNT(*) FROM (SELECT LOWER(TRIM(email)) k FROM customer_staging GROUP BY 1 HAVING COUNT(*)>1) x",
 'Q53 duplicate events': "SELECT COUNT(*) FROM (SELECT user_id,event_type,event_ts FROM events GROUP BY 1,2,3 HAVING COUNT(*)>1) x",
 'Q55 multiple payments per order': "SELECT COUNT(*) FROM (SELECT order_id FROM payments GROUP BY 1 HAVING COUNT(*)>1) x",
 'Q57 duplicate normalized SKUs': "SELECT COUNT(*) FROM (SELECT UPPER(TRIM(sku)) k FROM product_staging GROUP BY 1 HAVING COUNT(*)>1) x",
 'Q59 duplicate logs': "SELECT COUNT(*) FROM (SELECT pipeline_name,log_level,message,logged_at FROM pipeline_logs GROUP BY 1,2,3,4 HAVING COUNT(*)>1) x",
 'Q74 mismatched order totals': "SELECT COUNT(*) FROM (SELECT o.order_id FROM orders o JOIN order_items oi USING(order_id) GROUP BY o.order_id,o.total_amount HAVING ABS(o.total_amount-SUM(oi.quantity*oi.unit_price))>0.01) x",
 'Q77 SLA breaches': "SELECT COUNT(*) FROM support_tickets WHERE resolved_at IS NOT NULL AND EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0 > CASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END",
 'Q79 active low-stock products': "SELECT COUNT(*) FROM products WHERE active AND stock_quantity<reorder_level",
 'Q80 suspicious transaction groups': "SELECT COUNT(*) FROM (SELECT customer_id,transaction_ts::date FROM transactions GROUP BY 1,2 HAVING COUNT(*)>=3 AND SUM(amount)>1000) x",
}
with conn.cursor() as cur:
    edge_results=[]
    for name, query in checks.items():
        cur.execute(query)
        count=cur.fetchone()[0]
        edge_results.append({'check':name,'matching_rows':count,'pass':count>0})
edge_df=pd.DataFrame(edge_results)
display(edge_df)
assert edge_df['pass'].all(), 'One or more guaranteed edge cases are missing.'


In [ ]:
REFERENCE_SOLUTIONS = [('Q01 — Customer Order Summary', "SELECT c.customer_id, c.first_name || ' ' || c.last_name AS full_name,\n       COUNT(o.order_id) AS number_of_orders, SUM(o.total_amount) AS total_order_amount\nFROM customers c JOIN orders o ON o.customer_id=c.customer_id\nGROUP BY c.customer_id, c.first_name, c.last_name;"), ('Q02 — Customers Without Orders', "SELECT c.customer_id, c.first_name || ' ' || c.last_name AS full_name, c.city, c.signup_date\nFROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id\nWHERE o.order_id IS NULL;"), ('Q03 — Orders With Customer and Employee', "SELECT o.order_id,o.order_date,c.first_name||' '||c.last_name customer_name,\ne.first_name||' '||e.last_name employee_name,o.total_amount\nFROM orders o JOIN customers c ON c.customer_id=o.customer_id\nLEFT JOIN employees e ON e.employee_id=o.sales_employee_id;"), ('Q04 — Employees and Managers', "SELECT e.employee_id,e.first_name||' '||e.last_name employee_name,\nm.first_name||' '||m.last_name manager_name\nFROM employees e LEFT JOIN employees m ON m.employee_id=e.manager_id;"), ('Q05 — Products Never Ordered', 'SELECT p.product_id,p.product_name,p.category,p.unit_price\nFROM products p LEFT JOIN order_items oi ON oi.product_id=p.product_id\nWHERE oi.order_item_id IS NULL;'), ('Q06 — Customer Support Tickets', "SELECT t.ticket_id,c.first_name||' '||c.last_name customer_name,\ne.first_name||' '||e.last_name assigned_employee,t.status,t.priority\nFROM support_tickets t JOIN customers c ON c.customer_id=t.customer_id\nLEFT JOIN employees e ON e.employee_id=t.assigned_employee_id;"), ('Q07 — Order Line Details', 'SELECT oi.order_id,p.product_name,oi.quantity,oi.unit_price,\noi.quantity*oi.unit_price AS line_total\nFROM order_items oi JOIN products p ON p.product_id=oi.product_id;'), ('Q08 — Same-City Customers', "SELECT c1.customer_id,c1.first_name||' '||c1.last_name customer1,\nc2.customer_id,c2.first_name||' '||c2.last_name customer2,c1.city\nFROM customers c1 JOIN customers c2 ON c1.city=c2.city AND c1.customer_id<c2.customer_id;"), ('Q09 — Projects and Team Members', "SELECT p.project_id,p.project_name,e.employee_id,e.first_name||' '||e.last_name employee_name,\npa.role_on_project,pa.allocation_pct\nFROM projects p LEFT JOIN project_assignments pa ON pa.project_id=p.project_id\nLEFT JOIN employees e ON e.employee_id=pa.employee_id;"), ('Q10 — Orders With Payment Status', "SELECT o.order_id,c.first_name||' '||c.last_name customer_name,p.payment_status\nFROM orders o JOIN customers c ON c.customer_id=o.customer_id\nLEFT JOIN payments p ON p.order_id=o.order_id;"), ('Q11 — Orders by Status', 'SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY order_count DESC;'), ('Q12 — Revenue by Customer', "SELECT c.customer_id,c.first_name||' '||c.last_name full_name,SUM(o.total_amount) completed_revenue\nFROM customers c JOIN orders o ON o.customer_id=c.customer_id\nWHERE o.status='Completed' GROUP BY c.customer_id,c.first_name,c.last_name\nHAVING SUM(o.total_amount)>2000 ORDER BY completed_revenue DESC;"), ('Q13 — Department Salary Statistics', 'SELECT d.department_name,COUNT(e.employee_id) employee_count,MIN(e.salary) min_salary,\nMAX(e.salary) max_salary,ROUND(AVG(e.salary),2) avg_salary\nFROM departments d LEFT JOIN employees e ON e.department_id=d.department_id\nGROUP BY d.department_id,d.department_name;'), ('Q14 — Monthly Revenue', "SELECT DATE_TRUNC('month',order_date)::date AS month,SUM(total_amount) revenue\nFROM orders WHERE status='Completed' GROUP BY 1 ORDER BY 1;"), ('Q15 — Conditional Order Counts', "SELECT c.customer_id,COUNT(o.order_id) total_orders,\nCOUNT(*) FILTER (WHERE o.status='Completed') completed_orders,\nCOUNT(*) FILTER (WHERE o.status='Cancelled') cancelled_orders,\nCOUNT(*) FILTER (WHERE o.status='Pending') pending_orders\nFROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id GROUP BY c.customer_id;"), ('Q16 — Product Sales Performance', 'SELECT p.product_id,p.product_name,COALESCE(SUM(oi.quantity),0) units_sold,\nCOUNT(oi.order_item_id) order_lines,COALESCE(SUM(oi.quantity*oi.unit_price),0) sales_revenue\nFROM products p LEFT JOIN order_items oi ON oi.product_id=p.product_id GROUP BY p.product_id,p.product_name;'), ('Q17 — High-Volume Cities', 'SELECT city,COUNT(*) customer_count FROM customers GROUP BY city HAVING COUNT(*)>=3 ORDER BY customer_count DESC;'), ('Q18 — Ticket Status Summary', "SELECT priority,COUNT(*) total_tickets,\nCOUNT(*) FILTER (WHERE status='Open') open_tickets,\nCOUNT(*) FILTER (WHERE status='Resolved') resolved_tickets,\nCOUNT(*) FILTER (WHERE status='Closed') closed_tickets\nFROM support_tickets GROUP BY priority;"), ('Q19 — Average Order Value by Segment', 'SELECT c.segment,COUNT(o.order_id) order_count,COALESCE(SUM(o.total_amount),0) total_revenue,\nROUND(AVG(o.total_amount),2) average_order_value\nFROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id GROUP BY c.segment;'), ('Q20 — Department Project Allocation', 'SELECT d.department_name,COUNT(DISTINCT pa.employee_id) assigned_employees,\nROUND(AVG(pa.allocation_pct),2) avg_allocation\nFROM departments d LEFT JOIN employees e ON e.department_id=d.department_id\nLEFT JOIN project_assignments pa ON pa.employee_id=e.employee_id GROUP BY d.department_id,d.department_name;'), ('Q21 — Latest Order per Customer', 'WITH x AS (SELECT o.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date DESC,order_id DESC) rn FROM orders o)\nSELECT * FROM x WHERE rn=1;'), ('Q22 — Salary Rank by Department', 'SELECT employee_id,department_id,first_name,last_name,salary,\nRANK() OVER(PARTITION BY department_id ORDER BY salary DESC) salary_rank FROM employees;'), ('Q23 — Dense Salary Rank', 'SELECT employee_id,department_id,first_name,last_name,salary,\nDENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) dense_salary_rank FROM employees;'), ('Q24 — Previous Customer Order', 'SELECT customer_id,order_id,order_date,total_amount,\nLAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) previous_order_date,\nLAG(total_amount) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) previous_order_amount FROM orders;'), ('Q25 — Next Event Time', 'SELECT user_id,event_id,event_type,event_ts,\nLEAD(event_ts) OVER(PARTITION BY user_id ORDER BY event_ts,event_id) next_event_ts FROM events;'), ('Q26 — Running Customer Spend', 'SELECT customer_id,order_id,order_date,total_amount,\nSUM(total_amount) OVER(PARTITION BY customer_id ORDER BY order_date,order_id ROWS UNBOUNDED PRECEDING) running_spend FROM orders;'), ('Q27 — Department Salary vs Average', 'SELECT employee_id,department_id,salary,\nROUND(AVG(salary) OVER(PARTITION BY department_id),2) department_avg,\nROUND(salary-AVG(salary) OVER(PARTITION BY department_id),2) difference FROM employees;'), ('Q28 — Order Sequence Number', 'SELECT customer_id,order_id,order_date,\nROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date,order_id) order_sequence FROM orders;'), ('Q29 — Running Product Units', 'WITH d AS (SELECT oi.product_id,o.order_date,SUM(oi.quantity) units\nFROM order_items oi JOIN orders o ON o.order_id=oi.order_id GROUP BY oi.product_id,o.order_date)\nSELECT *,SUM(units) OVER(PARTITION BY product_id ORDER BY order_date ROWS UNBOUNDED PRECEDING) cumulative_units FROM d;'), ('Q30 — Customer Order Gap', 'WITH x AS (SELECT o.*,LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) prev_date FROM orders o)\nSELECT customer_id,order_id,order_date,prev_date,order_date-prev_date AS days_since_previous FROM x;'), ('Q31 — Above-Average Salary', 'SELECT * FROM employees WHERE salary>(SELECT AVG(salary) FROM employees) ORDER BY salary DESC;'), ('Q32 — Above Department Average', 'WITH a AS (SELECT department_id,AVG(salary) avg_salary FROM employees GROUP BY department_id)\nSELECT e.*,a.avg_salary FROM employees e JOIN a USING(department_id) WHERE e.salary>a.avg_salary;'), ('Q33 — Customers Above Average Spend', 'WITH s AS (SELECT customer_id,SUM(total_amount) spend FROM orders GROUP BY customer_id)\nSELECT c.customer_id,c.first_name,c.last_name,s.spend FROM s JOIN customers c USING(customer_id)\nWHERE s.spend>(SELECT AVG(spend) FROM s) ORDER BY s.spend DESC;'), ('Q34 — Best-Selling Products', 'WITH u AS (SELECT p.product_id,p.product_name,COALESCE(SUM(oi.quantity),0) units\nFROM products p LEFT JOIN order_items oi USING(product_id) GROUP BY p.product_id,p.product_name)\nSELECT * FROM u WHERE units>(SELECT AVG(units) FROM u) ORDER BY units DESC;'), ('Q35 — Orders Above Customer Average', 'SELECT o.* FROM orders o WHERE total_amount>(SELECT AVG(o2.total_amount) FROM orders o2 WHERE o2.customer_id=o.customer_id);'), ('Q36 — Departments With High Earners', 'SELECT d.* FROM departments d WHERE EXISTS\n(SELECT 1 FROM employees e WHERE e.department_id=d.department_id AND e.salary>110000);'), ('Q37 — Customers With No Support Tickets', 'SELECT c.* FROM customers c WHERE NOT EXISTS (SELECT 1 FROM support_tickets t WHERE t.customer_id=c.customer_id);'), ('Q38 — Project Staffing Pipeline', 'WITH s AS (SELECT project_id,COUNT(DISTINCT employee_id) headcount,SUM(allocation_pct) total_allocation\nFROM project_assignments GROUP BY project_id)\nSELECT p.project_id,p.project_name,s.headcount,s.total_allocation FROM projects p JOIN s USING(project_id) WHERE s.headcount>=3;'), ('Q39 — Revenue Share by Category', 'WITH c AS (SELECT p.category,SUM(oi.quantity*oi.unit_price) revenue FROM order_items oi JOIN products p USING(product_id) GROUP BY p.category),\nt AS (SELECT SUM(revenue) total FROM c)\nSELECT c.*,ROUND(100.0*c.revenue/NULLIF(t.total,0),2) revenue_pct FROM c CROSS JOIN t ORDER BY revenue DESC;'), ('Q40 — Repeat Customers', 'WITH x AS (SELECT customer_id,COUNT(*) order_count,MIN(order_date) first_order_date,MAX(order_date) latest_order_date,SUM(total_amount) lifetime_order_value\nFROM orders GROUP BY customer_id HAVING COUNT(*)>=3)\nSELECT c.customer_id,c.first_name,c.last_name,x.* FROM x JOIN customers c USING(customer_id);'), ('Q41 — Orders in Last 30 Days of Dataset', 'SELECT * FROM orders WHERE order_date BETWEEN (SELECT MAX(order_date)-29 FROM orders) AND (SELECT MAX(order_date) FROM orders) ORDER BY order_date;'), ('Q42 — Monthly Order Trend', "SELECT DATE_TRUNC('month',order_date)::date AS month,COUNT(*) order_count,SUM(total_amount) total_revenue FROM orders GROUP BY 1 ORDER BY 1;"), ('Q43 — Month-over-Month Revenue Change', "WITH m AS (SELECT DATE_TRUNC('month',order_date)::date AS month,SUM(total_amount) revenue FROM orders GROUP BY 1),\nx AS (SELECT *,LAG(revenue) OVER(ORDER BY month) previous_month_revenue FROM m)\nSELECT *,revenue-previous_month_revenue absolute_change,\nROUND(100.0*(revenue-previous_month_revenue)/NULLIF(previous_month_revenue,0),2) percentage_change FROM x ORDER BY month;"), ('Q44 — Latest Event per User', 'WITH x AS (SELECT e.*,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY event_ts DESC,event_id DESC) rn FROM events e)\nSELECT * FROM x WHERE rn=1;'), ('Q45 — Daily Active Users', 'SELECT event_ts::date AS event_date,COUNT(DISTINCT user_id) daily_active_users FROM events GROUP BY 1 ORDER BY 1;'), ('Q46 — Consecutive Login Days', "WITH d AS (SELECT DISTINCT user_id,event_ts::date AS login_date FROM events WHERE event_type='login')\nSELECT d1.user_id,d1.login_date FROM d d1 JOIN d d2 ON d2.user_id=d1.user_id AND d2.login_date=d1.login_date-1 ORDER BY 1,2;"), ('Q47 — Days to Resolve Tickets', "SELECT ticket_id,created_at,resolved_at,EXTRACT(EPOCH FROM(resolved_at-created_at))/86400.0 days_to_resolve\nFROM support_tickets WHERE status IN('Resolved','Closed') AND resolved_at IS NOT NULL;"), ('Q48 — Average Resolution Time by Priority', 'SELECT priority,ROUND(AVG(EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0),2) avg_resolution_hours\nFROM support_tickets WHERE resolved_at IS NOT NULL GROUP BY priority;'), ('Q49 — Customers Inactive for 60 Days', 'SELECT c.customer_id,c.first_name,c.last_name,MAX(o.order_date) latest_order\nFROM customers c LEFT JOIN orders o USING(customer_id) GROUP BY c.customer_id,c.first_name,c.last_name\nHAVING MAX(o.order_date) IS NULL OR MAX(o.order_date)<(SELECT MAX(order_date)-60 FROM orders);'), ('Q50 — Quarterly Revenue', "SELECT EXTRACT(YEAR FROM order_date)::int AS year,EXTRACT(QUARTER FROM order_date)::int AS quarter,\nCOUNT(*) order_count,SUM(total_amount) revenue FROM orders WHERE status='Completed' GROUP BY 1,2 ORDER BY 1,2;"), ('Q51 — Duplicate Customer Emails', 'SELECT LOWER(TRIM(email)) normalized_email,COUNT(*) copies FROM customer_staging GROUP BY 1 HAVING COUNT(*)>1;'), ('Q52 — Keep Latest Customer Record', 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY LOWER(TRIM(email)) ORDER BY updated_at DESC,staging_id DESC) rn FROM customer_staging)\nSELECT * FROM x WHERE rn=1;'), ('Q53 — Duplicate Events', 'SELECT user_id,event_type,event_ts,COUNT(*) copies FROM events GROUP BY user_id,event_type,event_ts HAVING COUNT(*)>1;'), ('Q54 — Remove Duplicate Events in Result', 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id,event_type,event_ts ORDER BY event_id DESC) rn FROM events)\nSELECT * FROM x WHERE rn=1;'), ('Q55 — Duplicate Payments', 'SELECT order_id,COUNT(*) payment_rows FROM payments GROUP BY order_id HAVING COUNT(*)>1;'), ('Q56 — Latest Payment per Order', 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY payment_date DESC,payment_id DESC) rn FROM payments)\nSELECT * FROM x WHERE rn=1;'), ('Q57 — Duplicate Product SKUs', 'SELECT UPPER(TRIM(sku)) normalized_sku,COUNT(*) copies FROM product_staging GROUP BY 1 HAVING COUNT(*)>1;'), ('Q58 — Canonical Product Staging', 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY UPPER(TRIM(sku)) ORDER BY updated_at DESC,staging_id DESC) rn FROM product_staging)\nSELECT * FROM x WHERE rn=1;'), ('Q59 — Duplicate Daily Logs', 'SELECT pipeline_name,log_level,message,logged_at,COUNT(*) copies FROM pipeline_logs\nGROUP BY pipeline_name,log_level,message,logged_at HAVING COUNT(*)>1;'), ('Q60 — Deduplicated Log Feed', 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY pipeline_name,log_level,message,logged_at ORDER BY log_id DESC) rn FROM pipeline_logs)\nSELECT * FROM x WHERE rn=1;'), ('Q61 — Top 3 Customers by Revenue', "SELECT c.customer_id,c.first_name,c.last_name,SUM(o.total_amount) revenue\nFROM customers c JOIN orders o USING(customer_id) WHERE o.status='Completed'\nGROUP BY c.customer_id,c.first_name,c.last_name ORDER BY revenue DESC LIMIT 3;"), ('Q62 — Second-Highest Company Salary', 'SELECT * FROM employees WHERE salary=(SELECT MAX(salary) FROM employees WHERE salary<(SELECT MAX(salary) FROM employees));'), ('Q63 — Top 3 Salaries per Department', 'WITH x AS (SELECT e.*,DENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) rnk FROM employees e)\nSELECT * FROM x WHERE rnk<=3 ORDER BY department_id,rnk;'), ('Q64 — Latest Transaction per Customer', 'WITH x AS (SELECT t.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY transaction_ts DESC,transaction_id DESC) rn FROM transactions t)\nSELECT * FROM x WHERE rn=1;'), ('Q65 — Top 2 Products per Category', 'WITH s AS (SELECT p.product_id,p.product_name,p.category,COALESCE(SUM(oi.quantity),0) units\nFROM products p LEFT JOIN order_items oi USING(product_id) GROUP BY p.product_id,p.product_name,p.category),\nr AS (SELECT *,DENSE_RANK() OVER(PARTITION BY category ORDER BY units DESC) rnk FROM s)\nSELECT * FROM r WHERE rnk<=2 ORDER BY category,rnk;'), ('Q66 — Highest-Value Order per Month', "WITH x AS (SELECT o.*,DENSE_RANK() OVER(PARTITION BY DATE_TRUNC('month',order_date) ORDER BY total_amount DESC) rnk FROM orders o)\nSELECT * FROM x WHERE rnk=1;"), ('Q67 — Top 3 Cities by Revenue', 'SELECT c.city,SUM(o.total_amount) revenue FROM customers c JOIN orders o USING(customer_id)\nGROUP BY c.city ORDER BY revenue DESC LIMIT 3;'), ('Q68 — Most Active Users', 'SELECT user_id,COUNT(*) event_count FROM events GROUP BY user_id ORDER BY event_count DESC LIMIT 5;'), ('Q69 — Third-Latest Order per Customer', 'WITH x AS (SELECT o.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date DESC,order_id DESC) rn FROM orders o)\nSELECT * FROM x WHERE rn=3;'), ('Q70 — Top Employee by Project Load', 'WITH x AS (SELECT employee_id,SUM(allocation_pct) total_allocation FROM project_assignments GROUP BY employee_id),\nr AS (SELECT *,DENSE_RANK() OVER(ORDER BY total_allocation DESC) rnk FROM x)\nSELECT e.employee_id,e.first_name,e.last_name,r.total_allocation FROM r JOIN employees e USING(employee_id) WHERE r.rnk=1;'), ('Q71 — Customer 360 Snapshot', "WITH o AS (SELECT customer_id,COUNT(*) lifetime_orders,SUM(total_amount) lifetime_revenue,MAX(order_date) latest_order_date FROM orders GROUP BY customer_id),\nt AS (SELECT customer_id,COUNT(*) FILTER (WHERE status='Open') open_ticket_count FROM support_tickets GROUP BY customer_id)\nSELECT c.customer_id,c.signup_date,c.segment,COALESCE(o.lifetime_orders,0) lifetime_orders,COALESCE(o.lifetime_revenue,0) lifetime_revenue,\no.latest_order_date,COALESCE(t.open_ticket_count,0) open_ticket_count FROM customers c LEFT JOIN o USING(customer_id) LEFT JOIN t USING(customer_id);"), ('Q72 — Potentially Stuck Pipelines', "WITH x AS (SELECT pipeline_name,logged_at::date AS run_date,COUNT(*) FILTER (WHERE log_level='ERROR') error_count,\nEXTRACT(EPOCH FROM(MAX(logged_at)-MIN(logged_at)))/60 duration_minutes FROM pipeline_logs GROUP BY pipeline_name,logged_at::date)\nSELECT * FROM x WHERE error_count>0 OR duration_minutes>90 ORDER BY run_date,pipeline_name;"), ('Q73 — Checkout Conversion Funnel', "SELECT COUNT(DISTINCT user_id) FILTER (WHERE event_type='product_view') viewed_users,\nCOUNT(DISTINCT user_id) FILTER (WHERE event_type='add_to_cart') cart_users,\nCOUNT(DISTINCT user_id) FILTER (WHERE event_type='purchase') purchased_users,\nROUND(100.0*COUNT(DISTINCT user_id) FILTER (WHERE event_type='purchase')/\nNULLIF(COUNT(DISTINCT user_id) FILTER (WHERE event_type='product_view'),0),2) view_to_purchase_pct FROM events;"), ('Q74 — Revenue Reconciliation', 'SELECT o.order_id,o.total_amount,SUM(oi.quantity*oi.unit_price) item_total,\no.total_amount-SUM(oi.quantity*oi.unit_price) difference\nFROM orders o JOIN order_items oi USING(order_id) GROUP BY o.order_id,o.total_amount\nHAVING ABS(o.total_amount-SUM(oi.quantity*oi.unit_price))>0.01;'), ('Q75 — Customer Churn Candidates', 'SELECT c.customer_id,c.first_name,c.last_name,COUNT(o.order_id) historical_orders,MAX(o.order_date) latest_order\nFROM customers c JOIN orders o USING(customer_id) GROUP BY c.customer_id,c.first_name,c.last_name\nHAVING COUNT(o.order_id)>=2 AND MAX(o.order_date)<(SELECT MAX(order_date)-90 FROM orders);'), ('Q76 — Payment Failure Rate', "SELECT DATE_TRUNC('month',payment_date)::date AS month,COUNT(*) total_attempts,\nCOUNT(*) FILTER (WHERE payment_status='Failed') failed_payments,\nROUND(100.0*COUNT(*) FILTER (WHERE payment_status='Failed')/COUNT(*),2) failure_rate_pct\nFROM payments GROUP BY 1 ORDER BY 1;"), ('Q77 — Support SLA Breaches', "SELECT ticket_id,priority,created_at,resolved_at,\nROUND(EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0,2) resolution_hours,\nCASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END sla_hours\nFROM support_tickets WHERE resolved_at IS NOT NULL\nAND EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0 >\nCASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END;"), ('Q78 — Pipeline Error Trend', "SELECT pipeline_name,logged_at::date AS log_date,COUNT(*) total_logs,\nCOUNT(*) FILTER (WHERE log_level='ERROR') error_logs,\nROUND(100.0*COUNT(*) FILTER (WHERE log_level='ERROR')/COUNT(*),2) error_rate_pct\nFROM pipeline_logs GROUP BY pipeline_name,logged_at::date ORDER BY log_date,pipeline_name;"), ('Q79 — Product Reorder Candidates', 'WITH mx AS (SELECT MAX(order_date) max_date FROM orders),\ns AS (SELECT oi.product_id,SUM(oi.quantity) units_30d FROM order_items oi JOIN orders o USING(order_id),mx\nWHERE o.order_date BETWEEN mx.max_date-29 AND mx.max_date GROUP BY oi.product_id)\nSELECT p.product_id,p.product_name,p.stock_quantity,p.reorder_level,COALESCE(s.units_30d,0) units_sold_last_30d\nFROM products p LEFT JOIN s USING(product_id) WHERE p.active AND p.stock_quantity<p.reorder_level;'), ('Q80 — Suspicious Transaction Pattern', 'SELECT customer_id,transaction_ts::date AS transaction_date,COUNT(*) transaction_count,SUM(amount) daily_amount\nFROM transactions GROUP BY customer_id,transaction_ts::date HAVING COUNT(*)>=3 AND SUM(amount)>1000 ORDER BY daily_amount DESC;')]
print(f"Loaded {len(REFERENCE_SOLUTIONS)} solutions")


In [ ]:
results=[]
for number,(title,query) in enumerate(REFERENCE_SOLUTIONS,1):
    try:
        with conn.cursor() as cur:
            cur.execute(query)
            # fetchmany(1) avoids transferring large result sets.
            first=cur.fetchmany(1) if cur.description else []
        status='PASS' if first else 'EMPTY'
        results.append({'question':f'Q{number:02}', 'title':title, 'status':status, 'error':''})
    except Exception as exc:
        conn.rollback()
        results.append({'question':f'Q{number:02}', 'title':title, 'status':'ERROR', 'error':str(exc)})
report=pd.DataFrame(results)
display(report)
print('Summary:',report.status.value_counts().to_dict())
report.to_csv('sql_interview_validation_report.csv',index=False)
print('Saved: sql_interview_validation_report.csv')
assert len(report)==80
assert (report.status=='PASS').all(), 'Review EMPTY/ERROR rows and their SQL; do not assume all 80 are verified.'


## Notes

- `PASS` means the query executed and returned at least one row; `EMPTY` means it executed but returned none.
- This notebook tests the 80 **original solution SQL statements** rather than substituting easier queries.
- For deterministic validation, keep the seeded data unchanged between reset and validation.
- If your database username differs, edit the connection cell.
